# 같은 세 프레임에서 가까운 쪽 관절 다시 찍기

같은 사람이 같은 원본을 두 번 보고 기준점을 얼마나 비슷하게 짚는지 확인합니다. 2차 판독 동안에는 1차 표시와 모델 예측을 숨깁니다. 지난 좌표가 기억난다면 시간을 조금 두고 진행해도 됩니다.

**Run All**로 편집기를 여세요. 각 탭에서 선수의 가까운 쪽 **어깨·골반·무릎·발목**을 판독합니다. 무릎은 관절선, 발목은 복사뼈, 골반은 고관절 소켓의 화면상 위치, c04 어깨는 견봉을 기준으로 합니다. 어떤 점을 볼 수 있는지는 이번 원본을 보고 다시 결정하세요.

- 위치를 짚을 수 있으면 해당 관절을 선택하고 사진을 클릭합니다. 근거를 두고 추정한 점은 **가림 추정점으로 변경**을 누릅니다. 골반 소켓은 자동으로 추정점이 됩니다.
- 좌표를 정할 수 없으면 **가려서 판독 불가** 또는 **위치 불확실**을 누릅니다. 두 상태 모두 판독을 마친 것으로 셉니다.
- 클릭할 때마다 2차 기록에 저장됩니다. 중간에 닫아도 이 노트북을 다시 열면 이번 회차의 점부터 이어갑니다.

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from wlcv.annotation.pose_points import JOINTS, load_points, pose_point_editor, pose_point_path
from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS
from wlcv.evaluation.pose import compare_pose_reads

samples = [("a03_sn_f_01", 19), ("b02_sn_h_02", 36), ("c04_sn_f_01", 78)]
split = pd.read_csv(SPLIT)
validation = split.loc[split["role"] == "validation", ["video_id", "person_id", "role"]]
metadata = pd.read_csv(ROOT / "data/metadata/videos.csv")
videos = validation.merge(metadata[["video_id", "filename", "near_side"]], on="video_id",
                          validate="one_to_one").set_index("video_id")

In [ ]:
editors = []
for video, frame in samples:
    info = videos.loc[video]
    editor = pose_point_editor(VIDEOS / info.filename, frame, info.person_id, info.near_side,
                               split=info.role, pass_number=2)
    editors.append(editor)

tabs = widgets.Tab(children=editors)
for index, (video, frame) in enumerate(samples):
    tabs.set_title(index, f"{video} · {frame}번")
display(tabs)

## 12개를 판독한 뒤 이 셀만 다시 실행하기

직접 표시한 점은 두 좌표 사이의 거리를 원본 픽셀로 봅니다. 한 번이라도 추정한 점은 참고 거리 열에 표시하고, 한 번이라도 판독 불가라면 상태만 비교합니다. 세 장의 원본 해상도와 촬영 배율이 다르므로, 우선 같은 프레임과 같은 관절 안에서 해석하세요.

In [ ]:
second_reads = {}
remaining = 0
for video, frame in samples:
    path = pose_point_path(ANNOTATIONS, video, frame, pass_number=2)
    records = load_points(path)
    second_reads[video, frame] = records
    remaining += sum(joint not in records for joint in JOINTS)

if remaining:
    print(f"2차 판독이 {remaining}개 남았습니다. 마치면 이 셀만 다시 실행하세요.")
else:
    for video, frame in samples:
        first = load_points(pose_point_path(ANNOTATIONS, video, frame))
        table = compare_pose_reads(first, second_reads[video, frame])
        print(f"{video} · {frame}번")
        display(table.style.format(precision=1, na_rep="—"))